# EASE baseline для контеста по рекомендациям

> **Embarrassingly Shallow Autoencoders for Sparse Data (EASE)** — сильный и очень интерпретируемый baseline для implicit-feedback рекомендаций.

В этом ноутбуке строится полный pipeline для того же контеста:

1. загрузка и проверка лога `user × track`;
2. агрегация повторных взаимодействий;
3. воспроизводимое train/validation-разбиение;
4. построение user–item матрицы;
5. обучение EASE с L2-регуляризацией;
6. подбор `lambda` по `NDCG@10`;
7. финальное обучение на всех warm-start данных;
8. генерация детерминированного submission.

### Главная идея EASE

Пусть `X` — матрица взаимодействий пользователей с объектами размера `n_users × n_items`. EASE ищет линейное восстановление каждого объекта через остальные:

$
\min_B \|X - XB\|_F^2 + \lambda \|B\|_F^2,
\quad \text{при } diag(B)=0.
$

Закрытое решение строится через матрицу Грама:

$
G = X^T X + \lambda I, \qquad P = G^{-1},
$
$
B_{ij} = -\frac{P_{ij}}{P_{jj}}, \qquad B_{ii}=0.
$

После обучения score кандидата вычисляется как `X_u B`. Нулевая диагональ принципиальна: уже наблюдавшийся трек не должен восстанавливать сам себя напрямую.

> **Важно:** EASE создаёт плотную матрицу `B` размера `n_items × n_items`. Это осознанный trade-off: метод быстрый и сильный на умеренном количестве items, но требует заметной памяти. В ноутбуке предсказания считаются блоками, поэтому не создаётся дополнительная плотная матрица scores для всех пользователей сразу.

## 1. Импорты и конфигурация

Ноутбук не устанавливает пакеты автоматически: это позволяет запускать его одинаково локально и в CI. Минимальные зависимости:

```bash
pip install pandas numpy scipy matplotlib seaborn
```

`scipy` используется для sparse user–item матрицы, `numpy` — для линейной алгебры EASE.

In [ ]:
from pathlib import Path
import gc
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

SEED = 31337
VALID_SIZE = 0.20
POSITIVE_THRESHOLD = 0.50
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 3
TOP_K = 10
USER_BLOCK_SIZE = 256

# Полный перебор lambda обычно дорогой для dense inverse. Сетка компактная и логарифмическая.
LAMBDA_GRID = [1_000.0, 3_000.0, 10_000.0, 15_000.0, 30_000.0]
RUN_LAMBDA_SEARCH = True
DEFAULT_LAMBDA = 15_000.0

DATA_DIR = Path("data/contest_1/raw_data")
OUTPUT_DIR = Path("data/contest_1/processed")
TRAIN_PATH = DATA_DIR / "train.csv"
SUBMISSION_INPUT_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = OUTPUT_DIR / "submit_ease.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)
np.random.seed(SEED)

print(f"Seed: {SEED}")
print(f"Train: {TRAIN_PATH.resolve()}")
print(f"Submission: {SUBMISSION_PATH.resolve()}")

## 2. Загрузка и проверка исходного лога

Ожидаемый формат `train.csv` — столбцы `user`, `track`, `time`. Мы не используем идентификаторы как числа: в исходных данных `user` — hex-подобные строки, а `track` может быть числовым идентификатором, поэтому приводим обе колонки к строковому типу.

In [ ]:
def validate_frame(frame: pd.DataFrame, required_columns: set[str]) -> None:
    missing = required_columns.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if frame.empty:
        raise ValueError("Input frame is empty")
    if frame[list(required_columns)].isna().any().any():
        raise ValueError("Required columns contain missing values")

raw = pd.read_csv(TRAIN_PATH)
validate_frame(raw, {"user", "track", "time"})
raw = raw[["user", "track", "time"]].copy()
raw["user"] = raw["user"].astype(str)
raw["track"] = raw["track"].astype(str)
raw["time"] = pd.to_numeric(raw["time"], errors="raise").astype("float32")

if (raw["time"] < 0).any():
    raise ValueError("Column 'time' must be non-negative")

print(f"Raw rows: {len(raw):,}")
print(f"Unique users: {raw['user'].nunique():,}")
print(f"Unique tracks: {raw['track'].nunique():,}")
raw.head()

## 3. Агрегация и формирование implicit feedback

Лог может содержать несколько событий для одной пары. Суммируем `time` и сохраняем число событий `play_count`. Для EASE нужна прежде всего структура user–item матрицы, поэтому основной вариант ниже — бинарный сигнал: пара считается положительной, если `total_time` превышает порог.

Такое решение соответствует implicit recommendation: модель учится отличать наблюдавшиеся взаимодействия от ненаблюдавшихся, а не предсказывать точное время прослушивания.

In [ ]:
grouped = (
    raw.groupby(["user", "track"], as_index=False)
       .agg(total_time=("time", "sum"), play_count=("time", "size"))
)

grouped = grouped.loc[grouped["total_time"] > POSITIVE_THRESHOLD].copy()
grouped["log_time"] = np.log1p(grouped["total_time"]).astype("float32")
grouped.reset_index(drop=True, inplace=True)

print(f"Unique pairs after threshold: {len(grouped):,}")
print(f"Users after threshold: {grouped['user'].nunique():,}")
print(f"Tracks after threshold: {grouped['track'].nunique():,}")
grouped.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(grouped["total_time"], bins=50, kde=True, ax=axes[0], color="#4472C4")
axes[0].set_title("Суммарное время взаимодействия")
axes[0].set_xlabel("total_time")

sns.histplot(grouped["log_time"], bins=50, kde=True, ax=axes[1], color="#ED7D31")
axes[1].set_title("Логарифмированный сигнал")
axes[1].set_xlabel("log1p(total_time)")
plt.tight_layout()
plt.show()

grouped["total_time"].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]).to_frame()

## 4. Воспроизводимый hold-out split

Сначала фиксируем split по парам, затем определяем warm-start пользователей и треки **только по train**. Это важно: если отфильтровать объекты по полной таблице до split, статистика validation начинает влиять на пространство обучения.

Ограничения `5` взаимодействий на пользователя и `3` на трек оставляем в качестве явного warm-start протокола исходного baseline. В метрике учитываются только validation-пары, для которых пользователь и трек известны train-модели.

In [ ]:
split_rng = np.random.default_rng(SEED)
grouped["is_validation"] = split_rng.random(len(grouped)) < VALID_SIZE

train_candidates = grouped.loc[~grouped["is_validation"]].copy()
valid_candidates = grouped.loc[grouped["is_validation"]].copy()

user_counts = train_candidates["user"].value_counts()
item_counts = train_candidates["track"].value_counts()
known_users = set(user_counts[user_counts >= MIN_USER_INTERACTIONS].index)
known_items = set(item_counts[item_counts >= MIN_ITEM_INTERACTIONS].index)

train_df = train_candidates.loc[
    train_candidates["user"].isin(known_users)
    & train_candidates["track"].isin(known_items)
].copy()
valid_df = valid_candidates.loc[
    valid_candidates["user"].isin(known_users)
    & valid_candidates["track"].isin(known_items)
].copy()

# Валидация должна содержать только пользователей с train-профилем.
valid_df = valid_df.loc[valid_df["user"].isin(set(train_df["user"]))].copy()

split_report = pd.DataFrame({
    "rows": [len(grouped), len(train_candidates), len(valid_candidates), len(train_df), len(valid_df)],
    "users": [grouped["user"].nunique(), train_candidates["user"].nunique(), valid_candidates["user"].nunique(), train_df["user"].nunique(), valid_df["user"].nunique()],
    "items": [grouped["track"].nunique(), train_candidates["track"].nunique(), valid_candidates["track"].nunique(), train_df["track"].nunique(), valid_df["track"].nunique()],
}, index=["all", "train_candidates", "valid_candidates", "train_warm_start", "valid_warm_start"])
split_report

## 5. User–item матрица

Для EASE строим бинарную CSR-матрицу `X`: одна положительная пара соответствует единице. Дубликаты безопасно агрегируются через `sum_duplicates`, а итоговая матрица переводится в `float32` для экономии памяти.

Порядок индексов фиксируется сортировкой идентификаторов. Это делает результаты и submission воспроизводимыми между запусками.

In [ ]:
users = sorted(train_df["user"].unique())
items = sorted(train_df["track"].unique())
user_to_idx = {user: idx for idx, user in enumerate(users)}
item_to_idx = {item: idx for idx, item in enumerate(items)}


def create_binary_matrix(frame: pd.DataFrame) -> sp.csr_matrix:
    rows = frame["user"].map(user_to_idx)
    cols = frame["track"].map(item_to_idx)
    mask = rows.notna() & cols.notna()
    matrix = sp.csr_matrix(
        (
            np.ones(int(mask.sum()), dtype=np.float32),
            (rows.loc[mask].astype(np.int32), cols.loc[mask].astype(np.int32)),
        ),
        shape=(len(users), len(items)),
        dtype=np.float32,
    )
    matrix.sum_duplicates()
    matrix.data[:] = 1.0
    return matrix

train_matrix = create_binary_matrix(train_df)
valid_matrix = create_binary_matrix(valid_df)

print(f"Train matrix shape: {train_matrix.shape}")
print(f"Train nnz: {train_matrix.nnz:,}")
print(f"Validation nnz: {valid_matrix.nnz:,}")
print(f"Train density: {train_matrix.nnz / np.prod(train_matrix.shape):.6%}")

## 6. Реализация EASE

### Почему регуляризация обязательна

`XᵀX` может быть плохо обусловлена: популярные треки имеют большие диагональные значения, а похожие треки создают сильную корреляцию. Добавление `lambda I` стабилизирует решение и контролирует сложность item–item связей.

### Численная реализация

Вместо явного вызова `np.linalg.inv(G)` используем `np.linalg.solve(G, I)`. Математически это то же получение `G⁻¹`, но `solve` является стандартной более устойчивой операцией линейной алгебры. Поскольку EASE по определению создаёт dense-матрицу, перед запуском стоит оценить память: для `m` items одна матрица `float64` занимает примерно `8m²` байт.

In [ ]:
def fit_ease(X: sp.csr_matrix, lambda_: float = 15_000.0) -> np.ndarray:
    """Fit EASE and return dense item-item coefficient matrix B."""
    if lambda_ <= 0:
        raise ValueError("lambda_ must be positive")

    n_items = X.shape[1]
    gram = (X.T @ X).toarray().astype(np.float64, copy=False)
    gram.flat[:: n_items + 1] += lambda_

    identity = np.eye(n_items, dtype=np.float64)
    precision = np.linalg.solve(gram, identity)

    diagonal = np.diag(precision).copy()
    if np.any(diagonal == 0):
        raise FloatingPointError("Zero diagonal in precision matrix")

    B = -precision / diagonal[np.newaxis, :]
    np.fill_diagonal(B, 0.0)
    return B.astype(np.float32, copy=False)


def estimate_dense_gb(n_items: int, dtype=np.float64) -> float:
    return n_items * n_items * np.dtype(dtype).itemsize / 1024**3

print(f"Approximate B memory (float64): {estimate_dense_gb(len(items)):.2f} GB")
print(f"Approximate B memory (float32): {estimate_dense_gb(len(items), np.float32):.2f} GB")

## 7. Ranking metrics и блочные предсказания

Основная метрика — `NDCG@10`: релевантные validation-треки получают больший вклад, если попадают выше в списке. Уже известные train-треки исключаем из кандидатов, иначе модель будет оцениваться на повторной рекомендации.

Блочный расчёт нужен по двум причинам:

1. не материализовать `n_users × n_items` scores целиком;
2. одинаково использовать код для validation и submission.

In [ ]:
def top_k_indices(values: np.ndarray, k: int) -> np.ndarray:
    k = min(k, values.size)
    if k == 0:
        return np.empty(0, dtype=np.int64)
    candidates = np.argpartition(values, -k)[-k:]
    return candidates[np.argsort(values[candidates])[::-1]]


def evaluate_ease(B: np.ndarray, X_train: sp.csr_matrix, X_valid: sp.csr_matrix, k: int = 10, block_size: int = 256) -> dict:
    ndcg_values, precision_values, recall_values = [], [], []
    valid_users = 0
    ideal_cache = {}

    for start in range(0, X_train.shape[0], block_size):
        stop = min(start + block_size, X_train.shape[0])
        score_block = (X_train[start:stop] @ B).astype(np.float32, copy=False)

        for local_user, user_idx in enumerate(range(start, stop)):
            relevant = X_valid[user_idx].indices
            if relevant.size == 0:
                continue
            valid_users += 1
            score_row = score_block[local_user]
            known = X_train[user_idx].indices
            score_row[known] = -np.inf
            ranked = top_k_indices(score_row, k)
            hits = np.isin(ranked, relevant)
            n_hits = int(hits.sum())

            precision_values.append(n_hits / max(len(ranked), 1))
            recall_values.append(n_hits / len(relevant))
            gains = hits.astype(np.float64) / np.log2(np.arange(2, len(ranked) + 2))
            dcg = gains.sum()
            n_relevant_at_k = min(len(relevant), k)
            if n_relevant_at_k not in ideal_cache:
                ideal_cache[n_relevant_at_k] = (1.0 / np.log2(np.arange(2, n_relevant_at_k + 2))).sum()
            ndcg_values.append(dcg / ideal_cache[n_relevant_at_k] if ideal_cache[n_relevant_at_k] else 0.0)

    if valid_users == 0:
        raise ValueError("No users with validation interactions")
    return {
        "ndcg@10": float(np.mean(ndcg_values)),
        "precision@10": float(np.mean(precision_values)),
        "recall@10": float(np.mean(recall_values)),
        "evaluated_users": valid_users,
    }

## 8. Подбор регуляризации по validation

`lambda` — главный гиперпараметр EASE. Малое значение позволяет модели подгоняться под корреляции в train и повышает риск переобучения; большое значение сильнее сглаживает item–item связи и приближает решение к popularity-like baseline.

Подбираем `lambda` по `NDCG@10`, а не по абсолютному значению score: для ranking quality важен порядок кандидатов.

In [ ]:
lambda_results = []
if RUN_LAMBDA_SEARCH:
    for lambda_value in LAMBDA_GRID:
        print(f"Fitting EASE, lambda={lambda_value:,.0f}")
        candidate_B = fit_ease(train_matrix, lambda_=lambda_value)
        metrics = evaluate_ease(candidate_B, train_matrix, valid_matrix, k=TOP_K, block_size=USER_BLOCK_SIZE)
        lambda_results.append({"lambda": lambda_value, **metrics})
        del candidate_B
        gc.collect()
else:
    lambda_results.append({"lambda": DEFAULT_LAMBDA})

lambda_results_df = pd.DataFrame(lambda_results)
if "ndcg@10" in lambda_results_df:
    best_lambda = float(lambda_results_df.loc[lambda_results_df["ndcg@10"].idxmax(), "lambda"])
else:
    best_lambda = DEFAULT_LAMBDA

print(f"Selected lambda: {best_lambda:,.0f}")
lambda_results_df

In [ ]:
if "ndcg@10" in lambda_results_df:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.lineplot(data=lambda_results_df, x="lambda", y="ndcg@10", marker="o", ax=ax, color="#4472C4")
    ax.set_xscale("log")
    ax.set_title("Validation NDCG@10 по регуляризации")
    ax.set_xlabel("lambda (log scale)")
    ax.set_ylabel("NDCG@10")
    plt.tight_layout()
    plt.show()

## 9. Финальное обучение

После выбора `lambda` нельзя использовать validation для ещё одного сравнения моделей. Вместо этого объединяем train и validation, строим новую полную warm-start матрицу и обучаем EASE с уже зафиксированным гиперпараметром.

Число пользователей и items может немного измениться относительно tuning-матрицы, поэтому отображения индексов создаём заново.

In [ ]:
all_warm = pd.concat([train_df, valid_df], ignore_index=True)
final_users = sorted(all_warm["user"].unique())
final_items = sorted(all_warm["track"].unique())
final_user_to_idx = {user: idx for idx, user in enumerate(final_users)}
final_item_to_idx = {item: idx for idx, item in enumerate(final_items)}


def create_final_matrix(frame: pd.DataFrame) -> sp.csr_matrix:
    rows = frame["user"].map(final_user_to_idx).to_numpy(dtype=np.int32)
    cols = frame["track"].map(final_item_to_idx).to_numpy(dtype=np.int32)
    matrix = sp.csr_matrix(
        (np.ones(len(frame), dtype=np.float32), (rows, cols)),
        shape=(len(final_users), len(final_items)),
        dtype=np.float32,
    )
    matrix.sum_duplicates()
    matrix.data[:] = 1.0
    return matrix

final_matrix = create_final_matrix(all_warm)
print(f"Final matrix shape: {final_matrix.shape}")
print(f"Final matrix nnz: {final_matrix.nnz:,}")
print(f"Final B memory (float32): {estimate_dense_gb(len(final_items), np.float32):.2f} GB")

B_final = fit_ease(final_matrix, lambda_=best_lambda)
print("Final EASE model fitted")

## 10. Формирование submission без случайных scores

`test.csv` содержит пары, для которых нужно выдать score. Известные пользователи и треки получают EASE score. Для cold-start строк используется детерминированный fallback по популярности трека в полном warm-start train: случайный шум здесь неуместен, потому что он делает одинаковый запуск невоспроизводимым.

Исходный порядок и дубликаты `test.csv` сохраняются. Это безопаснее, чем сначала делать `drop_duplicates`, а затем восстанавливать строки merge-ом.

In [ ]:
submission_input = pd.read_csv(SUBMISSION_INPUT_PATH)
validate_frame(submission_input, {"user", "track"})
submission_input = submission_input[["user", "track"]].copy()
submission_input["user"] = submission_input["user"].astype(str)
submission_input["track"] = submission_input["track"].astype(str)

# Популярность — только детерминированный fallback для unknown user/item.
item_popularity = all_warm["track"].value_counts(normalize=True)
submission_scores = np.full(len(submission_input), np.nan, dtype=np.float32)
known_mask = (
    submission_input["user"].isin(final_user_to_idx)
    & submission_input["track"].isin(final_item_to_idx)
)

for start in range(0, len(final_users), USER_BLOCK_SIZE):
    stop = min(start + USER_BLOCK_SIZE, len(final_users))
    block_users = final_users[start:stop]
    row_mask = known_mask & submission_input["user"].isin(block_users)
    if not row_mask.any():
        continue

    user_rows = np.array([final_user_to_idx[user] for user in submission_input.loc[row_mask, "user"]], dtype=np.int32)
    item_cols = np.array([final_item_to_idx[item] for item in submission_input.loc[row_mask, "track"]], dtype=np.int32)
    unique_rows, inverse = np.unique(user_rows, return_inverse=True)
    score_block = (final_matrix[unique_rows] @ B_final).astype(np.float32, copy=False)
    submission_scores[row_mask.to_numpy()] = score_block[inverse, item_cols]

fallback = submission_input["track"].map(item_popularity).fillna(0.0).to_numpy(dtype=np.float32)
submission_scores = np.where(np.isnan(submission_scores), fallback, submission_scores)

submission = submission_input.copy()
submission["score"] = submission_scores
submission.to_csv(SUBMISSION_PATH, index=False)

print(f"Rows written: {len(submission):,}")
print(f"Known user-track pairs: {known_mask.mean():.1%}")
print(f"Score range: [{submission['score'].min():.6f}, {submission['score'].max():.6f}]")
print(f"Saved: {SUBMISSION_PATH}")
submission.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(submission["score"], bins=50, ax=ax, color="#70AD47")
ax.set_title("Распределение submission scores")
ax.set_xlabel("EASE score / popularity fallback")
ax.set_ylabel("Rows")
plt.tight_layout()
plt.show()

## 11. Итоги и ограничения

### Что делает этот baseline качественным

- EASE описан математически, а не используется как black box.
- `lambda` выбирается по той же ranking-метрике, которая важна в контесте.
- train/validation protocol фиксирован и не использует статистику validation при построении train vocabulary.
- Предсказания и cold-start fallback детерминированы.
- Dense-операции ограничены блочным расчётом scores; память для основной EASE-матрицы оценивается заранее.

### Ограничения

1. В датасете нет явного времени события, поэтому split случайный, не temporal.
2. EASE не использует контентные признаки и не решает cold-start так хорошо, как hybrid-модель.
3. Плотная item–item матрица ограничивает число items, с которым метод комфортно работает.
4. Validation NDCG — оценка выбранного протокола, а не гарантия leaderboard score.

### Возможные следующие шаги

- сравнить бинарный сигнал с `log1p(total_time)` как sample weight;
- добавить popularity blend к EASE score;
- проверить несколько seed и более устойчивый user-level split;
- сравнить с LightFM/ALS и сделать ансамбль по rank-normalized scores.